# HW2: Word2Vector
## Start

In [ ]:
from idlelib.help_about import version
%run CBOW.ipynb
# start word2vec train process from here
txt_path = "data/"
save_vec_path = "model/word2vec/"
save_model_path = "model/"
train_path = "data/train/"

In [ ]:
# load text
en_text = load_data(txt_path, version='en')
# create corpus
en_corpus = preprocess_text(en_text, language='en')
# create vocab
en_vocab = build_vocab(en_corpus)

In [ ]:
# create training data
en_train_data = create_training_data(en_corpus, en_vocab, window_size=5)
torch.save(en_train_data, train_path+"en_train_data.pth")
# load train data
en_train_data = torch.load(train_path+"en_train_data.pth")
en_idx2word = {v:k for k,v in en_vocab.items()}
print(en_train_data[0], '\n', [en_idx2word.get(idx) for idx in en_train_data[0][0]], '\n', en_idx2word.get(en_train_data[0][1]))

In [ ]:
# train en model
model_en = CBOW(vocab_size=len(en_vocab), embedding_size=200).to(device)
# optimizer & loss function
optimizer, criterion = get_optim_and_loss(model_en, "SGD", "cross")
print(model_en, optimizer, criterion)

In [ ]:
epochs = 1000
# set train mode
model_en.train()
# start train
lossv_en = []
print('Start en training...')
train(model_en, en_train_data, optimizer, criterion, epochs, lossv_en, batch_size=32)
# plot 
plt.plot([epoch + 1 for epoch in range(0, epochs)], lossv_en)
plt.title('CBOW Training Loss (en)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
# save
save_model(model_en, save_model_path, version='en')
save_word_vectors(model_en, en_vocab, save_vec_path, version='en')

## 7. An Example Task to Use Trained Word Vectors
### Compute similarity among words

In [ ]:
# load word vectors
word_vectors_en = load_word_vectors(save_vec_path, version="en")

top_n = 3
# find similar top n words
en_word = "three"
# english
top_similarities_en = find_similar_words(en_word, word_vectors_en, top_n)
print(f"Most similar top {top_n} words to '{en_word}':")
for word, similarity in top_similarities_en:
    print(f"{word}: {similarity:.3f}")

In [ ]:
# plot
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(model_en.embedding.weight.cpu().detach().numpy())

word2ReduceDimensionVec = {}
for word in en_vocab.keys():
    word2ReduceDimensionVec[word] = principalComponents[en_vocab[word], :]
    
plt.figure(figsize=(10, 10))
count = 0
for word, wordvec in word2ReduceDimensionVec.items():
    if count < 100:
        plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
        plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号，否则负号会显示成方块
        plt.scatter(wordvec[0], wordvec[1])
        plt.annotate(word, (wordvec[0], wordvec[1]))
        count += 1
plt.show()